# A7. Two co-author checks, redone on the corrected-fire equations

(A) Is rain washout an alternative explanation for the PM2.5 equation? (Before the correction this asked whether TCWV in the old Eq. 1 was
rain scavenging; TCWV is no longer in the equation, so the question becomes: does a precipitation factor, or a TCWV factor, add anything to the
new Eq. 1, and does the equation hold on rain-free days?)  (B) Does the ozone result change if temperature enters as the daily maximum or the
afternoon mean instead of the daily mean?  Scoring: leave-one-year-out, constants refitted, pooled and mean-of-years R2, same rows within a test.

In [1]:
# Setup
import sys, glob
from pathlib import Path
HERE = Path.cwd().resolve()
sys.path.insert(0, str(HERE.parent)); sys.path.insert(0, str(HERE.parents[1]))
import warnings
import numpy as np, pandas as pd, xarray as xr
from scipy.optimize import curve_fit
from style import ROOT, DATA_OUT
warnings.filterwarnings("ignore")
P = ROOT / "Data/processed/colombia/bogota"

In [2]:
# Settings (edit here)
SAVE = True

In [3]:
# LOYO helper
def loyo(df, cols, mod, p0, y):
    X = tuple(df[c].to_numpy(float) for c in cols); yv = df[y].to_numpy(float); yr = df.date.dt.year.to_numpy()
    pf, _ = curve_fit(mod, X, yv, p0=p0, maxfev=50000); pred = np.full(len(df), np.nan)
    for k in np.unique(yr):
        te = yr == k
        try: p, _ = curve_fit(mod, tuple(a[~te] for a in X), yv[~te], p0=pf, maxfev=50000)
        except RuntimeError: p = pf
        pred[te] = mod(tuple(a[te] for a in X), *p)
    r2 = lambda m: 1 - ((yv[m] - pred[m]) ** 2).sum() / ((yv[m] - yv[m].mean()) ** 2).sum()
    per = [r2(yr == k) for k in np.unique(yr) if (yr == k).sum() >= 30]
    return dict(pooled=float(r2(np.ones(len(df), bool))), mean_of_years=float(np.mean(per)), constants=pf)

In [4]:
# A. PM2.5: rain
m = pd.read_parquet(P / "bogota_master_dataset.parquet"); m["date"] = pd.to_datetime(m["date"]); m = m.sort_values("date").reset_index(drop=True)
cols = ["co", "wind_speed_era5", "hotspots_sector_N", "hotspots_sector_E_lag2", "precip_total_mm", "precip_total_mm_lag1", "tcwv"]
a = m.dropna(subset=cols + ["pm25"]).reset_index(drop=True); print("A: N", len(a))
lead = lambda X, al, be: np.sqrt(np.clip(al * (X[3] + X[2]), 0, None)) + be * X[0] / np.sqrt(X[1])
lead_rain = lambda X, al, be, k: lead(X, al, be) * (1 + k * np.sqrt(X[4]))
lead_rain1 = lambda X, al, be, k: lead(X, al, be) * (1 + k * np.sqrt(X[5]))
lead_tcwv = lambda X, al, be, k: lead(X, al, be) * (1 + k * X[6] / 30)
rows = []
for nm, mod, p0 in [("lead (Eq. 1)", lead, [0.104, 15.68]), ("x (1 + k sqrt(precip, same day))", lead_rain, [0.104, 15.68, 0.0]),
                    ("x (1 + k sqrt(precip, day before))", lead_rain1, [0.104, 15.68, 0.0]), ("x (1 + k TCWV/30)", lead_tcwv, [0.104, 15.68, 0.0])]:
    r = loyo(a, cols, mod, p0, "pm25"); rows.append(dict(variant=nm, pooled=r["pooled"], mean_of_years=r["mean_of_years"], k=(r["constants"][-1] if "k" in nm else np.nan)))
print(pd.DataFrame(rows).round(4).to_string(index=False))
dry = (a.precip_total_mm < 1) & (a.precip_total_mm_lag1 < 1)
sub = []
for lab, s in (("dry days (< 1 mm today and yesterday)", a[dry]), ("all other days", a[~dry])):
    s = s.reset_index(drop=True); r = loyo(s, cols, lead, [0.104, 15.68], "pm25")
    sub.append(dict(subset=lab, n=len(s), pooled=r["pooled"], mean_of_years=r["mean_of_years"]))
fire_free = lambda X, al: al * np.sqrt(X[0] / X[1])
for lab, s in (("dry days", a[dry]), ("all other days", a[~dry])):
    s = s.reset_index(drop=True); r = loyo(s, cols, fire_free, [20.6], "pm25"); sub.append(dict(subset=lab + ", fire-free CO/wind skeleton", n=len(s), pooled=r["pooled"], mean_of_years=r["mean_of_years"]))
print(pd.DataFrame(sub).round(3).to_string(index=False))
rain_stats = dict(corr_pm25_precip_same_day=float(a.pm25.corr(a.precip_total_mm)), corr_pm25_precip_lag1=float(a.pm25.corr(a.precip_total_mm_lag1)), frac_dry_days=float(dry.mean()), n_dry=int(dry.sum()))
mon = a.groupby(a.date.dt.month).agg(pm25=("pm25", "mean"), precip=("precip_total_mm", "mean")).round(1)
print(rain_stats); print(mon.T.to_string())
if SAVE:
    pd.DataFrame(rows).to_csv(DATA_OUT / "pm25_rain_addone.csv", index=False); pd.DataFrame(sub).to_csv(DATA_OUT / "pm25_dry_days.csv", index=False)
    pd.DataFrame([rain_stats]).to_csv(DATA_OUT / "pm25_rain_stats.csv", index=False); mon.to_csv(DATA_OUT / "pm25_precip_monthly.csv")

A: N 3283
                           variant  pooled  mean_of_years       k
                      lead (Eq. 1)  0.6768         0.6548     NaN
  x (1 + k sqrt(precip, same day))  0.6745         0.6534 -0.0066
x (1 + k sqrt(precip, day before))  0.6753         0.6541 -0.0081
                 x (1 + k TCWV/30)  0.6886         0.6684 -0.3897
                                    subset    n  pooled  mean_of_years
     dry days (< 1 mm today and yesterday)  616   0.410          0.385
                            all other days 2667   0.680          0.653
      dry days, fire-free CO/wind skeleton  616   0.233          0.164
all other days, fire-free CO/wind skeleton 2667   0.448          0.415
{'corr_pm25_precip_same_day': 0.3842409947361597, 'corr_pm25_precip_lag1': 0.2925081048173672, 'frac_dry_days': 0.18763326226012794, 'n_dry': 616}
date      1     2     3     4     5     6     7     8     9     10    11    12
pm25    20.2  25.0  26.9  18.9  15.2  12.9  11.3  14.1  16.5  18.4  20.7  19.9


In [5]:
# B. Ozone: temperature as daily mean, daily maximum, afternoon mean
o = pd.read_parquet(P / "bogota_o3_master_dataset.parquet"); o["date"] = pd.to_datetime(o["date"])
T = [xr.open_dataset(f)["t2m"].mean(["latitude", "longitude"]).to_series() - 273.15 for f in sorted(glob.glob(str(ROOT / "Data/external/era5/bogota_era5_t2m_lts_*.nc")))]
t = pd.concat(T); t.index = t.index - pd.Timedelta(hours=5); g = t.groupby(t.index.date)
tm = pd.DataFrame({"t_era_mean": g.mean(), "tmax_era": g.max()}); aft = t[(t.index.hour >= 12) & (t.index.hour <= 16)]
tm["t_aft"] = aft.groupby(aft.index.date).mean(); tm.index = pd.to_datetime(tm.index)
o = o.merge(tm, left_on="date", right_index=True, how="left")
need = ["pm25", "o3", "no2", "no", "radiation_mean", "hotspots_sector_E_lag2", "temp_2m", "tmax_era", "t_era_mean", "t_aft"]
b = o.dropna(subset=need).reset_index(drop=True); print("B: N", len(b))
lead3 = lambda X, a_, b_: np.sqrt(np.clip(a_ * X[0] * np.sqrt(np.clip(X[3] * X[1] / X[2], 0, None)) + b_ * X[4], 0, None))
def with_T(X, a_, b_, k): return lead3(X[:5], a_, b_) * (1 + k * X[5])
base_cols = ["radiation_mean", "no2", "no", "pm25", "hotspots_sector_E_lag2"]
r0 = loyo(b, base_cols, lead3, [0.4916, 0.1079], "o3"); res = [dict(variant="lead (Eq. 2), no temperature", pooled=r0["pooled"], mean_of_years=r0["mean_of_years"], k_per_sd=np.nan)]
for c in ("temp_2m", "t_era_mean", "tmax_era", "t_aft"):
    b["_z"] = (b[c] - b[c].mean()) / b[c].std()
    r = loyo(b, base_cols + ["_z"], with_T, [0.4916, 0.1079, 0.0], "o3")
    res.append(dict(variant=f"lead x (1 + k z), z = standardised {c}", pooled=r["pooled"], mean_of_years=r["mean_of_years"], k_per_sd=float(r["constants"][-1])))
res = pd.DataFrame(res); print(res.round(4).to_string(index=False))
def resid(y, X): X = np.column_stack([np.ones(len(y))] + X); return y - X @ np.linalg.lstsq(X, y, rcond=None)[0]
pr = []
for c in (None, "temp_2m", "tmax_era"):
    X = [b.no2.values, b.no.values, b.radiation_mean.values] + ([b[c].values] if c else [])
    pr.append(dict(control_adds=c or "none", partial_r=float(np.corrcoef(resid(b.pm25.values, X), resid(b.o3.values, X))[0, 1])))
pr = pd.DataFrame(pr); print(pr.round(3).to_string(index=False))
print("corr PM2.5 with ERA5 Tmax", round(b.pm25.corr(b.tmax_era), 3), "with station T", round(b.pm25.corr(b.temp_2m), 3), "| corr station T - Tmax", round(b.temp_2m.corr(b.tmax_era), 3))
if SAVE: res.to_csv(DATA_OUT / "o3_temperature_variants.csv", index=False); pr.to_csv(DATA_OUT / "o3_partial_r_temperature_variants.csv", index=False)

B: N 2918
                                      variant  pooled  mean_of_years  k_per_sd
                 lead (Eq. 2), no temperature  0.6032         0.5369       NaN
   lead x (1 + k z), z = standardised temp_2m  0.6142         0.5494    0.0363
lead x (1 + k z), z = standardised t_era_mean  0.6056         0.5391    0.0167
  lead x (1 + k z), z = standardised tmax_era  0.6134         0.5488    0.0326
     lead x (1 + k z), z = standardised t_aft  0.6124         0.5479    0.0310


control_adds  partial_r
        none      0.512
     temp_2m      0.512
    tmax_era      0.520
corr PM2.5 with ERA5 Tmax 0.014 with station T -0.003 | corr station T - Tmax 0.421
